# Experiment 6: Sequence Prediction using LSTM

## Step 1
Import the required libraries.

In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader
import kagglehub

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 2
Load the dataset and reshape each image into a sequence of 28 rows with 28 values each.

In [2]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
csv_path = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)[0]
df = pd.read_csv(csv_path)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

max_vocab = 10000
max_len = 200
batch_size = 64


def tokenize(text):
    text = re.sub(r"<[^>]+>", " ", text.lower())
    return re.findall(r"[a-z']+", text)


n_test = int(0.2 * len(df))
n_val = int(0.1 * (len(df) - n_test))
test_df = df.iloc[len(df) - n_test:]
val_df = df.iloc[len(df) - n_test - n_val:len(df) - n_test]
train_df = df.iloc[:len(df) - n_test - n_val]

counter = Counter()
for t in train_df["review"]:
    counter.update(tokenize(t))
vocab = {w: i + 2 for i, (w, _) in enumerate(counter.most_common(max_vocab - 2))}


def encode(text):
    ids = [vocab.get(w, 1) for w in tokenize(text)][:max_len]
    return [0] * (max_len - len(ids)) + ids


def to_tensors(d):
    X = torch.tensor([encode(t) for t in d["review"]], dtype=torch.long)
    y = torch.tensor((d["sentiment"] == "positive").astype(int).values, dtype=torch.float32)
    return X, y


Xt, yt = to_tensors(train_df)
Xv, yv = to_tensors(val_df)
Xte, yte = to_tensors(test_df)


def make_loader(X, y, bs, shuffle):
    return DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=shuffle)


print(Xt.shape, Xv.shape, Xte.shape)

100%|██████████| 25.7M/25.7M [00:24<00:00, 1.10MB/s]

Extracting files...


torch.Size([36000, 200]) torch.Size([4000, 200]) torch.Size([10000, 200])


## Step 3
Define the two LSTM layers and the linear output layer.

In [3]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size=max_vocab, embed_dim=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, num_layers=2, batch_first=True, dropout=0.3)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, (h, c) = self.lstm(self.embedding(x))
        return self.fc(self.dropout(h[-1])).squeeze(1)


epochs = 5
use_amp = device.type == "cuda"
model = SentimentLSTM().to(device)
criterion = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"parameters: {sum(p.numel() for p in model.parameters())}")

parameters: 871553


## Step 4
Train the LSTM for 5 epochs to predict the next row from the previous rows.

In [4]:
def free_memory():
    if device.type == "cuda":
        torch.cuda.empty_cache()


def evaluate(model, X, y, bs=128):
    model.eval()
    correct = 0
    total_loss = 0.0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = X[i:i + bs].to(device)
            yb = y[i:i + bs].to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(xb)
            total_loss += criterion(logits.float(), yb).item() * len(yb)
            correct += ((logits > 0).float() == yb).sum().item()
    return total_loss / len(X), correct / len(X)


def train_one_epoch(bs):
    model.train()
    tot = 0.0
    n = 0
    for xb, yb in make_loader(Xt, yt, bs, True):
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(xb)
        loss = criterion(logits.float(), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        tot += loss.item() * len(yb)
        n += len(yb)
    return tot / n


for ep in range(epochs):
    while True:
        try:
            train_loss = train_one_epoch(batch_size)
            break
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True)
            free_memory()
            if batch_size <= 8:
                raise
            batch_size = max(8, batch_size // 2)
            print(f"out of memory, retrying epoch with batch_size {batch_size}")
    free_memory()
    val_loss, val_acc = evaluate(model, Xv, yv)
    print(f"epoch {ep + 1}/{epochs} batch_size {batch_size} train_loss {train_loss:.4f} val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

epoch 1/5 batch_size 64 train_loss 0.6051 val_loss 0.5259 val_acc 0.7582
epoch 2/5 batch_size 64 train_loss 0.4278 val_loss 0.3881 val_acc 0.8313
epoch 3/5 batch_size 64 train_loss 0.3225 val_loss 0.3246 val_acc 0.8662
epoch 4/5 batch_size 64 train_loss 0.2585 val_loss 0.3068 val_acc 0.8758
epoch 5/5 batch_size 64 train_loss 0.2139 val_loss 0.3466 val_acc 0.8718


## Step 5
Test the LSTM by predicting the last 14 rows from the first 14 rows, output the error against the baseline and save the picture.

In [5]:
free_memory()
test_loss, test_acc = evaluate(model, Xte, yte)
print(f"test loss: {test_loss:.4f}")
print(f"test accuracy: {test_acc:.4f}")

model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(Xte), 128):
        preds.append((model(Xte[i:i + 128].to(device)) > 0).float().cpu())
preds = torch.cat(preds)
tp = ((preds == 1) & (yte == 1)).sum().item()
tn = ((preds == 0) & (yte == 0)).sum().item()
fp = ((preds == 1) & (yte == 0)).sum().item()
fn = ((preds == 0) & (yte == 1)).sum().item()
print(f"true positive {tp} true negative {tn} false positive {fp} false negative {fn}")
print(f"precision {tp / max(tp + fp, 1):.4f} recall {tp / max(tp + fn, 1):.4f}")


def predict_review(text):
    x = torch.tensor([encode(text)], dtype=torch.long, device=device)
    with torch.no_grad():
        p = torch.sigmoid(model(x)).item()
    return ("positive" if p > 0.5 else "negative"), p


for r in ["This movie was great, I loved every minute of it", "Terrible film, boring and a complete waste of time"]:
    label, prob = predict_review(r)
    print(f"{label} ({prob:.3f}): {r}")

torch.save(model.state_dict(), "exp06_imdb_lstm.pt")

test loss: 0.3499
test accuracy: 0.8658
true positive 4536 true negative 4123 false positive 881 false negative 460
precision 0.8374 recall 0.9079
positive (0.995): This movie was great, I loved every minute of it
negative (0.005): Terrible film, boring and a complete waste of time
